In [ ]:
import os
import cv2
import shutil
import hashlib

from pathlib import Path

# Configuration
SRC_ROOT = "path/to/raw/human_face_emotions"        
DST_ROOT = "path/to/cleaned/human_face_emotions"    
TARGET_RESOLUTION = (224, 224)                     
TARGET_FORMAT = "jpg"                               
COLOR_CONVERT = cv2.COLOR_BGR2RGB                   

# Emotion classes (assumes folder names)
CLASSES = [d.name for d in Path(SRC_ROOT).iterdir() if d.is_dir()]


# Utility to compute file hash for duplicates
def file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, "rb") as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

# Step 1: Create clean folder structure
for emotion in CLASSES:
    dst_folder = Path(DST_ROOT) / emotion
    dst_folder.mkdir(parents=True, exist_ok=True)

# Step 2: Iterate images, check format, convert, resize, rename
seen_hashes = set()
for emotion in CLASSES:
    src_folder = Path(SRC_ROOT) / emotion
    for img_file in src_folder.iterdir():
        if img_file.is_file():
            try:
                # Compute hash to detect duplicates
                h = file_hash(img_file)
                if h in seen_hashes:
                    print(f"Skipping duplicate: {img_file}")
                    continue
                seen_hashes.add(h)

                # Read image
                img = cv2.imread(str(img_file))
                if img is None:
                    print(f"Corrupted / unreadable image: {img_file}")
                    continue

                # Convert color if needed (ensuring 3-channel)
                if len(img.shape) == 2 or img.shape[2] == 1:
                    img = cv2.cvtColor(img, COLOR_CONVERT)
                elif img.shape[2] == 4:
                    img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

                # Resize
                img = cv2.resize(img, TARGET_RESOLUTION, interpolation=cv2.INTER_AREA)

                # Build new filename
                new_name = f"{emotion.lower()}_{img_file.stem}.{TARGET_FORMAT}"
                dst_path = Path(DST_ROOT) / emotion / new_name

                # Save as jpg with quality setting
                cv2.imwrite(str(dst_path), img, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

            except Exception as e:
                print(f"Error processing {img_file}: {e}")

print("Preprocessing complete.")
